In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
from lsff_utils import paths, gbd_data
import pathlib

In [2]:
location = "India"
vehicle = "rice"
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "india"
vehicle = "wheat"


In [4]:
location = location.title()

In [5]:
# Simulation output lives in per-run, timestamp-named directories, so the runs
# are resolved rather than hardcoded. The child rescaling divides through by
# pregnancy outcome counts, so it reads the maternal run as well as the child one.
maternal_results = paths.latest_results(
    paths.source_root(paths.MATERNAL_RESULTS_ROOT, data_source), location, vehicle
)
child_results = paths.latest_results(
    paths.source_root(paths.CHILD_RESULTS_ROOT, data_source), location, vehicle
)
child_results

PosixPath('/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/child/wheat/india/2026_11_23_02_17_45/results')

In [6]:
with gbd_data.quiet_gbd_logs():
    pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2023        2024        1.936945e+05
                  0.019178   0.076712    2023        2024        5.766332e+05
                  0.076712   0.500000    2023        2024        4.257480e+06
                  0.500000   1.000000    2023        2024        5.021249e+06
                  1.000000   2.000000    2023        2024        1.024770e+07
                  2.000000   5.000000    2023        2024        3.231858e+07
                  5.000000   10.000000   2023        2024        5.835119e+07
                  10.000000  15.000000   2023        2024        6.286521e+07
                  15.000000  20.000000   2023        2024        6.432311e+07
                  20.000000  25.000000   2023        2024        6.374038e+07
                  25.000000  30.000000   2023        2024        6.159864e+07
                  30.000000  35.000000   2023        2024        5.733795e+07
  

In [7]:
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
    ).value
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end  parameter  
India     Female  10.0       15.0     2023        2024      lower_value    0.000159
                                                            mean_value     0.000349
                                                            upper_value    0.000705
                  15.0       20.0     2023        2024      lower_value    0.007324
                                                            mean_value     0.008912
                                                            upper_value    0.010701
                  20.0       25.0     2023        2024      lower_value    0.080282
                                                            mean_value     0.094863
                                                            upper_value    0.111437
                  25.0       30.0     2023        2024      lower_value    0.113628
                                                            mean_value     0.124175
    

In [8]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2023        2024        0.000000
                  0.019178   0.076712    2023        2024        0.000000
                  0.076712   0.500000    2023        2024        0.000000
                  0.500000   1.000000    2023        2024        0.000000
                  1.000000   2.000000    2023        2024        0.000000
                  2.000000   5.000000    2023        2024        0.000000
                  5.000000   10.000000   2023        2024        0.000000
                  10.000000  15.000000   2023        2024        0.000349
                  15.000000  20.000000   2023        2024        0.008912
                  20.000000  25.000000   2023        2024        0.094863
                  25.000000  30.000000   2023        2024        0.124175
                  30.000000  35.000000   2023        2024        0.079277
                  35.000000  40.000000   2023     

In [9]:
births = pop * asfr
births[births > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2023        2024        2.194007e+04
                  15.0       20.0     2023        2024        5.732639e+05
                  20.0       25.0     2023        2024        6.046632e+06
                  25.0       30.0     2023        2024        7.648997e+06
                  30.0       35.0     2023        2024        4.545600e+06
                  35.0       40.0     2023        2024        1.760489e+06
                  40.0       45.0     2023        2024        5.427846e+05
                  45.0       50.0     2023        2024        1.201325e+05
                  50.0       55.0     2023        2024        9.494763e+03
Name: value, dtype: float64

In [10]:
births = births.sum()
f"{int(births):,}"

'21,269,334'

In [11]:
sim_baseline_births = pd.read_parquet(
    paths.measure_path(maternal_results, "pregnancy_outcome_count")
)
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,partial_term,10_to_14,invalid,1,baseline,0,141,1.0
1,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,partial_term,10_to_14,invalid,2,baseline,0,141,3.0
2,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,partial_term,10_to_14,invalid,3,baseline,0,141,4.0
3,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,partial_term,10_to_14,invalid,4,baseline,0,141,1.0
4,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,partial_term,10_to_14,invalid,5,baseline,0,141,1.0
...,...,...,...,...,...,...,...,...,...,...,...
809995,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,stillbirth,95_plus,severe,1,zero,0,129,0.0
809996,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,stillbirth,95_plus,severe,2,zero,0,129,0.0
809997,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,stillbirth,95_plus,severe,3,zero,0,129,0.0
809998,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,stillbirth,95_plus,severe,4,zero,0,129,0.0


In [12]:
sim_baseline_births = sim_baseline_births[
    (sim_baseline_births.scenario == "baseline")
    & (sim_baseline_births.sub_entity == "live_birth")
]
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
5,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,10_to_14,invalid,1,baseline,0,141,10.0
6,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,10_to_14,invalid,2,baseline,0,141,6.0
7,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,10_to_14,invalid,3,baseline,0,141,12.0
8,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,10_to_14,invalid,4,baseline,0,141,10.0
9,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,10_to_14,invalid,5,baseline,0,141,7.0
...,...,...,...,...,...,...,...,...,...,...,...
807290,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,95_plus,severe,1,baseline,0,54,0.0
807291,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,95_plus,severe,2,baseline,0,54,0.0
807292,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,95_plus,severe,3,baseline,0,54,0.0
807293,pregnancy_outcome_count,custom_fertility,pregnancy_outcome,live_birth,95_plus,severe,4,baseline,0,54,0.0


In [13]:
sim_baseline_births = sim_baseline_births.groupby("input_draw").value.sum().mean()
sim_baseline_births

np.float64(8828903.0)

In [14]:
births

np.float64(21269334.701552395)

In [15]:
scalar = births / sim_baseline_births
scalar

np.float64(2.4090574674512104)

In [16]:
# NOTE: 'person_time' was renamed to 'person_time_population' when PersonTimeObserver
# became a PublicHealthObserver, so the automated V&V loader can discover it.
for result in ["ylds", "ylls", "deaths", "person_time_population"]:
    df = pd.read_parquet(
        paths.measure_path(child_results, result)
    )
    df.value *= scalar
    path = (
        paths.output_root(paths.RESCALED_CHILD_RESULTS_ROOT)
        / vehicle
        / location.lower()
        / f"{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)